# Mão na Roda — Fine-tuning do Albertina PT-BR
Diagnóstico Preliminar de Falhas Automotivas Baseado em IA

Notebook equivalente aos do BERTimbau, BERTabaporu, DeBERTinha e
JabuticaBERT, trocando o modelo base para o **Albertina PT-BR**
(`PORTULAN/albertina-100m-portuguese-ptbr-encoder`).

**Sobre o modelo:** desenvolvido por equipe conjunta da Universidade de
Lisboa e da Universidade do Porto, o Albertina é um encoder da família
BERT construído sobre a arquitetura **DeBERTa**, distribuído em três
portes (1,5B, 900M e 100M de parâmetros) e em duas variantes de
português (PT-PT europeu e PT-BR brasileiro). Este notebook usa a
variante **PT-BR de 100M de parâmetros** — a única das três que cabe
confortavelmente na GPU T4 gratuita do Colab, mantendo o porte
comparável aos demais modelos já testados no projeto.

**Por que esse experimento é relevante para o TCC:** com o Albertina,
o projeto passa a ter dois representantes de arquitetura DeBERTa (o
outro é o JabuticaBERT) e dois de arquitetura BERT (BERTimbau e
BERTabaporu), além de duas linhagens diferentes dentro da própria
família DeBERTa: JabuticaBERT (treinado do zero em português) e
Albertina/DeBERTinha (adaptados a partir de bases já existentes). Isso
fortalece a discussão sobre qual variável — arquitetura ou proveniência
do pré-treino — mais influencia o desempenho nesta tarefa específica de
classificação de falhas automotivas.

**Antes de rodar:** Ambiente de execução → Alterar tipo de ambiente de
execução → GPU (T4).

## Passo 1 — Setup e verificação do ambiente

Inclui a remoção preventiva do `torchvision`, mesma correção de bug de
ambiente aplicada nos notebooks anteriores.

In [2]:
# ============================================================
# PASSO 1 — Setup e verificação do ambiente
# Projeto Mão na Roda — Fine-tuning do Albertina PT-BR
# ============================================================

# --- 1.0 Corrige bug de ambiente conhecido (datasets x torchvision) ---
!pip uninstall -y torchvision -q

# --- 1.1 Instala as dependências ---
!pip install -q transformers datasets accelerate scikit-learn pandas openpyxl sentencepiece

# --- 1.2 Verifica se a GPU está disponível ---
import torch
print("Torch:", torch.__version__)
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  ATENÇÃO: nenhuma GPU detectada. Vá em 'Ambiente de execução' > "
          "'Alterar tipo de ambiente de execução' e selecione GPU antes de continuar.")

# --- 1.3 Constantes do projeto ---
# Único ponto que muda em relação aos notebooks anteriores: o checkpoint.
MODEL_CHECKPOINT = "PORTULAN/albertina-100m-portuguese-ptbr-encoder"  # Albertina PT-BR 100M
SEED = 42
NUM_LABELS = 10  # as 10 classes de falha do projeto
MAX_LENGTH = 64  # mesmo valor usado nos notebooks anteriores, para comparação justa

LABEL_NAMES = [
    "Bateria/Sistema Elétrico",       # 0
    "Sistema de Freios",              # 1
    "Superaquecimento do Motor",      # 2
    "Suspensão/Amortecedores",        # 3
    "Sistema de Transmissão/Câmbio",  # 4
    "Vazamento de Óleo",              # 5
    "Sistema de Arrefecimento",       # 6
    "Pneu/Roda",                      # 7
    "Sistema de Injeção/Combustível", # 8
    "Sistema de Escapamento",         # 9
]

import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print()
print("Setup concluído. Modelo base:", MODEL_CHECKPOINT)

# ============================================================
# --- 1.4 Carregando os arquivos de split (via Google Drive) ---
# ============================================================
# Reutiliza os MESMOS splits dos notebooks anteriores — essencial para
# que a comparação entre os 5 modelos seja válida.

from google.colab import drive
drive.mount('/content/drive')

CAMINHO_BASE = '/content/drive/MyDrive/Agente Mecânico/notebooks/Finetuning - 3000 relatos/3 Splits - Finetuning/Splits'

import pandas as pd
import os

CONFIGS = ["80_10_10", "70_15_15", "60_20_20"]

splits = {}
for tag in CONFIGS:
    splits[tag] = {
        "treino": pd.read_csv(os.path.join(CAMINHO_BASE, f"split_treino_{tag}.csv")),
        "val":    pd.read_csv(os.path.join(CAMINHO_BASE, f"split_val_{tag}.csv")),
        "teste":  pd.read_csv(os.path.join(CAMINHO_BASE, f"split_teste_{tag}.csv")),
    }
    n_tr, n_val, n_te = len(splits[tag]["treino"]), len(splits[tag]["val"]), len(splits[tag]["teste"])
    print(f"{tag}: treino={n_tr} | val={n_val} | teste={n_te} | total={n_tr+n_val+n_te}")

assert "peso_treino" in splits["80_10_10"]["treino"].columns, \
    "Coluna peso_treino não encontrada — confira se subiu o CSV certo (do v15, não uma versão antiga)."
print("\nColuna peso_treino presente. Pronto para o Passo 2 (tokenização).")

Torch: 2.11.0+cu128
GPU disponível: True
GPU: Tesla T4

Setup concluído. Modelo base: PORTULAN/albertina-100m-portuguese-ptbr-encoder


ValueError: mount failed

## Passo 2 — Tokenização

O tokenizer é carregado a partir do próprio `MODEL_CHECKPOINT`, como em
todos os notebooks anteriores.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_batch(batch):
    return tokenizer(
        batch["relato"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def preparar_dataset_hf(df):
    # Converte um DataFrame pandas em um Dataset do Hugging Face,
    # já tokenizado e com as colunas 'labels' e 'weight' prontas para o Trainer.
    ds = Dataset.from_pandas(df[["relato", "falha_label", "peso_treino"]].reset_index(drop=True))
    ds = ds.map(tokenize_batch, batched=True)
    ds = ds.rename_column("falha_label", "labels")
    ds = ds.rename_column("peso_treino", "weight")
    ds = ds.remove_columns(["relato"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "weight"])
    return ds

# Tokeniza as 3 configs de uma vez (usadas na comparação leve do Passo 4)
datasets_tokenizados = {}
for tag in CONFIGS:
    datasets_tokenizados[tag] = {
        "treino": preparar_dataset_hf(splits[tag]["treino"]),
        "val":    preparar_dataset_hf(splits[tag]["val"]),
        "teste":  preparar_dataset_hf(splits[tag]["teste"]),
    }
    print(f"{tag}: tokenização concluída "
          f"({len(datasets_tokenizados[tag]['treino'])} treino, "
          f"{len(datasets_tokenizados[tag]['val'])} val, "
          f"{len(datasets_tokenizados[tag]['teste'])} teste)")

# Checagem rápida: mostra um exemplo tokenizado para conferência visual
exemplo = datasets_tokenizados["80_10_10"]["treino"][0]
print("\nExemplo tokenizado:")
print("input_ids (primeiros 15):", exemplo["input_ids"][:15])
print("labels:", exemplo["labels"], "->", LABEL_NAMES[int(exemplo["labels"])])
print("weight:", exemplo["weight"])

# --- Checagem de taxa de truncamento em MAX_LENGTH=64 ---
def contar_truncados(df, tokenizer, max_length):
    truncados = 0
    for texto in df["relato"]:
        ids_completos = tokenizer(texto, truncation=False)["input_ids"]
        if len(ids_completos) > max_length:
            truncados += 1
    return truncados, len(df)

n_truncados, n_total = contar_truncados(splits["80_10_10"]["treino"], tokenizer, MAX_LENGTH)
print(f"\nRelatos truncados em MAX_LENGTH={MAX_LENGTH} (treino, Albertina PT-BR): "
      f"{n_truncados}/{n_total} ({100*n_truncados/n_total:.1f}%)")

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/963 [00:00<?, ?B/s]

... (8 outputs left out)

Map:   0%|          | 0/1824 [00:00<?, ? examples/s]

Map:   0%|          | 0/608 [00:00<?, ? examples/s]

Map:   0%|          | 0/609 [00:00<?, ? examples/s]

60_20_20: tokenização concluída (1824 treino, 608 val, 609 teste)

Exemplo tokenizado:
input_ids (primeiros 15): tensor([    1,   673,   512,  1001,   295,  4214,   784, 11742,   263,  4112,
         4842,   295, 15477,   783,     6])
labels: tensor(0) -> Bateria/Sistema Elétrico
weight: tensor(0.5000)

Relatos truncados em MAX_LENGTH=64 (treino, Albertina PT-BR): 0/2432 (0.0%)


## Passo 3 — `WeightedTrainer` e métricas

Idêntico aos notebooks anteriores.

In [ ]:
import torch.nn as nn
from transformers import Trainer
from sklearn.metrics import f1_score, accuracy_score, classification_report

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        weights = inputs.pop("weight").float()
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(reduction="none")
        per_example_loss = loss_fct(logits, labels)
        loss = (per_example_loss * weights).sum() / weights.sum()
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
    }

print("WeightedTrainer e compute_metrics definidos.")

WeightedTrainer e compute_metrics definidos.


## Passo 4 — Treino leve comparativo (3 configs)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments

def rodar_treino_leve(tag, epocas=3):
    model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS, torch_dtype=torch.float32
  )

    args = TrainingArguments(
        output_dir=f"./resultados_leve_albertina_{tag}",
        num_train_epochs=epocas,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        seed=SEED,
        remove_unused_columns=False,   # crítico — não remover
        report_to="none",
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=datasets_tokenizados[tag]["treino"],
        eval_dataset=datasets_tokenizados[tag]["val"],
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics_val = trainer.evaluate()
    return metrics_val["eval_f1_macro"], metrics_val["eval_accuracy"]

resultados_leve = {}
for tag in CONFIGS:
    print(f"\n{'='*60}\nTreino leve — config {tag}\n{'='*60}")
    f1, acc = rodar_treino_leve(tag, epocas=3)
    resultados_leve[tag] = {"f1_macro": f1, "accuracy": acc}
    print(f"{tag}: F1-macro (val) = {f1:.4f} | acurácia (val) = {acc:.4f}")

print("\n=== Comparação final (treino leve, Albertina PT-BR) ===")
for tag, m in sorted(resultados_leve.items(), key=lambda x: -x[1]["f1_macro"]):
    print(f"{tag}: F1-macro={m['f1_macro']:.4f}  acc={m['accuracy']:.4f}")

# Lembrete: diferenças de F1 abaixo de ~0.01 devem ser tratadas como empate
# técnico nesta escala de dataset.

## Passo 5 — Escolha da configuração vencedora

In [ ]:
# Ajustem esta linha de acordo com o resultado do Passo 4
CONFIG_VENCEDORA = "80_10_10"  # <-- editar se outra config vencer

print(f"Config escolhida para o treino final: {CONFIG_VENCEDORA}")
print(f"Treino: {len(datasets_tokenizados[CONFIG_VENCEDORA]['treino'])} exemplos")
print(f"Validação: {len(datasets_tokenizados[CONFIG_VENCEDORA]['val'])} exemplos")
print(f"Teste: {len(datasets_tokenizados[CONFIG_VENCEDORA]['teste'])} exemplos (fica intocado até a avaliação final)")

Config escolhida para o treino final: 80_10_10
Treino: 2432 exemplos
Validação: 305 exemplos
Teste: 304 exemplos (fica intocado até a avaliação final)


## Passo 6 — Treino completo com early stopping

Mesma estratégia dos notebooks anteriores: até 15 épocas, com early
stopping (patience=3) monitorando o F1-macro de validação.

In [ ]:
from transformers import EarlyStoppingCallback

model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS, torch_dtype=torch.float32
)
model_final = model_final.float()  # força TODOS os tensores (incluindo buffers internos) para float32

args_final = TrainingArguments(
    output_dir="./resultados_final_albertina",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="epoch",
    seed=SEED,
    remove_unused_columns=False,
    report_to="none",
    fp16=False,   # garante que não haja precisão mista automática
    bf16=False,   # garante que não haja precisão mista automática
)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] DebertaForSequenceClassification LOAD REPORT from: PORTULAN/albertina-100m-portuguese-ptbr-encoder
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downst

## Passo 7 — Avaliação final no conjunto de teste

In [ ]:
resultado_teste = trainer_final.evaluate(datasets_tokenizados[CONFIG_VENCEDORA]["teste"])
print("=== Resultado final no TESTE (Albertina PT-BR) ===")
print(f"F1-macro: {resultado_teste['eval_f1_macro']:.4f}")
print(f"Acurácia: {resultado_teste['eval_accuracy']:.4f}")

# Relatório por classe — essencial para comparar com os demais modelos,
# em especial no par Superaquecimento x Arrefecimento
preds_output = trainer_final.predict(datasets_tokenizados[CONFIG_VENCEDORA]["teste"])
preds = np.argmax(preds_output.predictions, axis=-1)
labels_reais = preds_output.label_ids

print("\n=== Relatório por classe ===")
print(classification_report(labels_reais, preds, target_names=LABEL_NAMES, digits=3))

AttributeError: `AcceleratorState` object has no attribute `distributed_type`. This happens if `AcceleratorState._reset_state()` was called and an `Accelerator` or `PartialState` was not reinitialized.

## Passo 8 — Salvar o modelo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CAMINHO_SALVAR = '/content/drive/MyDrive/Agente Mecânico/modelos/albertina_mao_na_roda/'

trainer_final.save_model(CAMINHO_SALVAR)
tokenizer.save_pretrained(CAMINHO_SALVAR)

print(f"Modelo e tokenizer salvos em: {CAMINHO_SALVAR}")
print("Conteúdo salvo:")
!ls -la "{CAMINHO_SALVAR}"

import json
resumo = {
    "modelo": MODEL_CHECKPOINT,
    "config_split": CONFIG_VENCEDORA,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "f1_macro_teste": float(resultado_teste["eval_f1_macro"]),
    "acuracia_teste": float(resultado_teste["eval_accuracy"]),
    "melhor_f1_validacao": float(trainer_final.state.best_metric),
}
with open(CAMINHO_SALVAR + "resumo_treino.json", "w", encoding="utf-8") as f:
    json.dump(resumo, f, ensure_ascii=False, indent=2)

print("\nResumo salvo em resumo_treino.json:")
print(json.dumps(resumo, ensure_ascii=False, indent=2))